# Closure test: $\chi\chi\to b\bar b$ against arXiv:2112.09706

The direct-annihilation arm of the GCE fit, run on the same measurement, covariance and halo as the
portal regions, and compared with the $b\bar b$ region of Fig. 18 of arXiv:2112.09706 (south ROI).
The templates are built at their $\rho_\odot = 0.4$ GeV/cm$^3$, so the red region (halo fixed) is a
like-for-like comparison with their fixed-halo contour. The blue one profiles $\rho_\odot$ over our
prior, $0.44 \pm 0.13$ GeV/cm$^3$, as everywhere else in the paper.

**Input:** the direct $b\bar b$ spectra written by `GCE/scripts/make_bb_reference.py` (run it once
first; the masses, event count and seed are imported from it), and the digitised contour in
`GCE/data/contours_2112.09706/`.

**Output:** `GCE/paper_figs/GCE_bb_closure_v0.{png,pdf}`

In [ ]:
import re
import sys
from pathlib import Path

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.gridspec import GridSpec
from matplotlib.legend_handler import HandlerTuple
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.ticker import LogLocator, NullFormatter
from scipy.ndimage import gaussian_filter1d
from scipy.ndimage import shift as ndimage_shift
from tqdm import trange

ROOT = Path.cwd().resolve()
while ROOT.name and not (ROOT / 'hidden_sector_DM').is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'GCE' / 'scripts'))    # the mass grid lives with its script

from GCE.source.halo import HALO_2112, RHO_LOCAL_2112, RHO_LOCAL_PRIOR
from GCE.source.likelihood import direct_template, gls_fit, measurement
from make_bb_reference import MASS_GRID, N_EVENTS, SEED

plt.style.use(ROOT / 'notebooks' / 'mine.mplstyle')
plt.rcParams.update({
    'text.usetex': True,
    'font.family': 'serif',
    'text.latex.preamble': r'\usepackage{amssymb}'
})
FIG_DIR = ROOT / 'GCE' / 'paper_figs'
FIG_DIR.mkdir(parents=True, exist_ok=True)


def color_shades(color, n, lightest_alpha=0.25):
    """n shades of `color` blended toward white, darkest first."""
    rgb = np.array(mcolors.to_rgb(color))
    fracs = np.linspace(1.0, lightest_alpha, n)
    return [mcolors.to_hex(rgb*f + (1-f)) for f in fracs]

## 1. Fit each mass
The 14-bin template of $\chi\chi\to b\bar b$ at each mass of `make_bb_reference.MASS_GRID`, built at
the 2112.09706 halo (our profile shape at their $\rho_\odot = 0.4$ GeV/cm$^3$) with $\kappa = 8$, their
self-conjugate convention, and $\langle\sigma v\rangle$ profiled out in closed form by `gls_fit`.

In [ ]:
REGION = 'south'        # 'both' (40x40), 'south' or 'north'
N_GCE_BINS = 14         # bins in the measurement
HALO_CLOSURE = dict(HALO_2112, rho_local=RHO_LOCAL_2112)    # 2112.09706's density, 0.4 GeV/cm^3

phi, Cinv = measurement(region=REGION)      # once; independent of the mass

n_mass = len(MASS_GRID)
sigmav = np.full(n_mass, np.nan)       # best-fit <sigma v> [cm^3/s]
sigmav_err = np.full(n_mass, np.nan)   # its 1-sigma error [cm^3/s]
chi2 = np.full(n_mass, np.nan)

for i in trange(n_mass, desc='b bbar'):
    t = direct_template(MASS_GRID[i], n_events=N_EVENTS, seed=SEED, halo=HALO_CLOSURE)
    sigmav[i], sigmav_err[i], chi2[i] = gls_fit(t, phi, Cinv)

# The region below reads chi2 off the GLS parabola, which holds only at an interior fit.
clamped = MASS_GRID[sigmav == 0.0]
if clamped.size:
    print(f'WARNING: <sigma v> clamped at 0 for m_DM = {clamped} GeV; their parabola is a '
          f'one-sided limit')

i_best = int(np.nanargmin(chi2))
print(f'best fit: m_DM = {MASS_GRID[i_best]:g} GeV, <sigma v> = {sigmav[i_best]:.3e} cm^3/s, '
      f'chi2 = {chi2[i_best]:.2f} / {N_GCE_BINS - 2} dof')

## 2. The $(m_{\rm DM}, \langle\sigma v\rangle)$ region
$\chi^2$ on a log $\langle\sigma v\rangle$ axis, the same construction `paper_figs.ipynb` uses for the
portals. At fixed halo it is the GLS parabola. With $\rho_\odot$ free, $\bar J \propto \rho_\odot^2$
only rescales $\langle\sigma v\rangle$, so each $\rho_\odot$ is a shift along the log axis (measured
from the templates' 0.4 GeV/cm$^3$), plus the log-normal penalty of our $0.44 \pm 0.13$ prior,
minimised over $\rho_\odot$.

In [ ]:
SV_AXIS = np.logspace(-27, -24, 700)    # <sigma v> [cm^3/s]
dln_sv = np.diff(np.log(SV_AXIS))[0]

# --- Fixed halo: chi2 is exactly parabolic in <sigma v> at an interior fit ---
chi2_fixed_halo = chi2[None, :] + ((SV_AXIS[:, None] - sigmav[None, :])/sigmav_err[None, :])**2

# --- rho_sun profiled: a shift along log <sigma v> plus the prior penalty ---
rho_mean, rho_sigma = RHO_LOCAL_PRIOR    # GeV/cm^3
rho_width = rho_sigma/rho_mean           # log-normal width
rhos = rho_mean*np.exp(np.linspace(-4, 4, 201)*rho_width)
ln_Jbar_shifts = 2.0*np.log(rhos/HALO_CLOSURE['rho_local'])    # relative to the templates' density
halo_penalties = (np.log(rhos/rho_mean)/rho_width)**2

chi2_profiled = np.full_like(chi2_fixed_halo, np.inf)
for u, penalty in zip(ln_Jbar_shifts, halo_penalties):
    shifted = ndimage_shift(chi2_fixed_halo, (-u/dln_sv, 0), order=1, cval=np.inf,
                            mode='constant')
    np.minimum(chi2_profiled, shifted + penalty, out=chi2_profiled)

# --- Delta chi2, lightly smoothed along m_DM ---
dchi2_fixed_halo = gaussian_filter1d(chi2_fixed_halo - np.nanmin(chi2_fixed_halo), 1, axis=1,
                                     mode='nearest')
dchi2_profiled = gaussian_filter1d(chi2_profiled - np.nanmin(chi2_profiled), 1, axis=1,
                                   mode='nearest')

## 3. The 2112.09706 contour
Fig. 18 of arXiv:2112.09706, south ROI, digitised: one block per confidence level, columns
$m_{\rm DM}$ [GeV] and $\langle\sigma v\rangle$ [$10^{-26}$ cm$^3$/s].

In [ ]:
def read_contours(path):
    """{68: (N, 2), 95: (N, 2)} closed contours of (m_DM [GeV], <sigma v> [1e-26 cm^3/s])."""
    out = {}
    key = None
    with open(path) as f:
        for line in f:
            m = re.match(r'\s*#\s*(\d+)\s*%', line)
            if m:
                key = int(m.group(1))
                out[key] = []
            elif line.strip() and not line.lstrip().startswith('#'):
                out[key].append(line.split())
    return {k: np.array(v + v[:1], float) for k, v in out.items()}


contours_2112 = read_contours(ROOT / 'GCE' / 'data' / 'contours_2112.09706'
                              / 'DM_to_bb_contour_GCE_south.txt')

## 4. Figure

In [ ]:
css = [color_shades('k', 2), color_shades('tab:red', 2), color_shades('tab:blue', 2)]
DCHI2_LEVELS = [0, 2.30, 6.18]     # 68% and 95% C.L. for two parameters

fig1 = plt.figure(figsize=(3.9, 3.9/1.61803398875*1.1))
gs1 = GridSpec(20, 20, figure=fig1)
ax1 = fig1.add_subplot(gs1[:, :])

# --- Fits: rho_sun profiled underneath, fixed halo on top ---
ax1.contourf(MASS_GRID, SV_AXIS, dchi2_profiled, levels=DCHI2_LEVELS,
             colors=[css[2][0], css[2][1]], zorder=2)
ax1.contourf(MASS_GRID, SV_AXIS, dchi2_fixed_halo, levels=DCHI2_LEVELS,
             colors=[css[1][0], css[1][1]], zorder=3)

# --- 2112.09706, Fig. 18 ---
for cl in (68, 95):
    ax1.plot(contours_2112[cl][:, 0], contours_2112[cl][:, 1]*1e-26, c='k', ls='--', lw=1,
             zorder=6)

ax1.set_yscale('log')
ax1.set_xlim(30, 52)
ax1.set_ylim(2e-27, 1e-25)
ax1.set_xticks([30, 35, 40, 45, 50])
ax1.set_yticks([1e-26, 1e-25])
ax1.yaxis.set_minor_locator(LogLocator(base=10.0, subs=np.arange(1, 10)*0.1, numticks=2000))
ax1.yaxis.set_minor_formatter(NullFormatter())
ax1.set_xlabel(r'$m_{\rm DM}\,\,[{\rm GeV}]$')
ax1.set_ylabel(r'$\langle\sigma v\rangle\,\,[{\rm cm}^3/{\rm s}]$')
ax1.text(51, 2.5e-27, r'$\chi\chi\rightarrow b\bar b$', fontsize=mpl.rcParams['legend.fontsize'],
         ha='right')

labels = ['2112.09706', r'GCE $\chi^2$ fit, $\rho_\odot = ' + f'{RHO_LOCAL_2112:g}' + r'$',
          r'$+\,\rho_\odot = ' + f'{rho_mean:g}' + r'\pm' + f'{rho_sigma:g}'
          + r'\,{\rm GeV}/{\rm cm}^3$']
handles = [Line2D([0], [0], c='k', ls='--', lw=1),
           (Patch(facecolor=css[1][0]), Patch(facecolor=css[1][1])),
           (Patch(facecolor=css[2][0]), Patch(facecolor=css[2][1]))]
ax1.legend(handles=handles, labels=labels, loc='lower left', handletextpad=0.4,
           handler_map={tuple: HandlerTuple(ndivide=None, pad=0.0)})

plt.savefig(FIG_DIR / 'GCE_bb_closure_v0.png')
plt.savefig(FIG_DIR / 'GCE_bb_closure_v0.pdf')
plt.show()
plt.close()